# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [14]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [20]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [43]:
links = fetch_website_links("https://notarec.fi/")
links

['#main',
 'https://notarec.fi',
 'https://notarec.fi/rekrytointi-hr-palvelut-hinta',
 'https://notarec.fi/referenssit',
 'https://notarec.fi/palvelut',
 'https://notarec.fi/rekrytointipalvelu',
 'https://notarec.fi/hr-palvelut',
 'https://notarec.fi/esihenkilovalmennukset',
 'https://notarec.fi/uravalmennus',
 'https://notarec.fi/rekrytointi-hr-palvelut',
 'https://notarec.fi/rekrytointi-hr/teknologiayritykset',
 'https://notarec.fi/rekrytointi-hr/asiantuntijayritykset',
 'https://notarec.fi/rekrytointi-hr-energia',
 'https://notarec.fi/rekrytointi-teollisuus',
 'https://notarec.fi/rekrytointi-tietoa',
 'https://notarec.fi/rekrytointi-tietoa',
 'https://notarec.fi/not-another-blog',
 'https://notarec.fi/tapahtumat-webinaarit',
 'https://notarec.fi/meista',
 'https://notarec.fi/yhteys',
 'https://notarec.fi/meista',
 'https://notarec.fi/notarec-liittyy-htgp-konserniin-uutta-osaamista-yritysten-kasvuun-notarec',
 'https://ura.notarec.fi',
 'https://notarec.fi/rekrytointi-hr-palvelut-hin

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [1]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:

print(get_links_user_prompt("https://notarec.fi/"))


Here is the list of links on the website https://notarec.fi/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
https://notarec.fi
https://notarec.fi/rekrytointi-hr-palvelut-hinta
https://notarec.fi/referenssit
https://notarec.fi/palvelut
https://notarec.fi/rekrytointipalvelu
https://notarec.fi/hr-palvelut
https://notarec.fi/esihenkilovalmennukset
https://notarec.fi/uravalmennus
https://notarec.fi/rekrytointi-hr-palvelut
https://notarec.fi/rekrytointi-hr/teknologiayritykset
https://notarec.fi/rekrytointi-hr/asiantuntijayritykset
https://notarec.fi/rekrytointi-hr-energia
https://notarec.fi/rekrytointi-teollisuus
https://notarec.fi/rekrytointi-tietoa
https://notarec.fi/rekrytointi-tietoa
https://notarec.fi/not-another-blog
https://notarec.fi/tapahtumat-webinaarit
https://notarec.fi/meista
https://notare

In [32]:
def select_relevant_links(url):
    print(f"select_relevant_links for {url} by calling the {MODEL}...")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links:")
    return links
    

In [29]:
select_relevant_links("https://notarec.fi/")

select_relevant_links for https://notarec.fi/ by calling the LLM...
Found 28 relevant links:


{'links': [{'type': 'company homepage', 'url': 'https://notarec.fi'},
  {'type': 'about page', 'url': 'https://notarec.fi/meista'},
  {'type': 'corporate info',
   'url': 'https://notarec.fi/notarec-liittyy-htgp-konserniin-uutta-osaamista-yritysten-kasvuun-notarec'},
  {'type': 'services page', 'url': 'https://notarec.fi/palvelut'},
  {'type': 'references', 'url': 'https://notarec.fi/referenssit'},
  {'type': 'customer story',
   'url': 'https://notarec.fi/asiakastarina-profilence?hsLang=fi'},
  {'type': 'customer story',
   'url': 'https://notarec.fi/asiakastarina-haltian?hsLang=fi'},
  {'type': 'customer story',
   'url': 'https://notarec.fi/asiakastarina-junnikkala?hsLang=fi'},
  {'type': 'customer story',
   'url': 'https://notarec.fi/asiakastarina-creowave?hsLang=fi'},
  {'type': 'HR services', 'url': 'https://notarec.fi/hr-palvelut'},
  {'type': 'HR services (2025)',
   'url': 'https://notarec.fi/hr-palvelut-2025?hsLang=fi'},
  {'type': 'HR recruitment',
   'url': 'https://notare

In [ ]:
select_relevant_links("https://notarec.fi/")

In [27]:
select_relevant_links("https://www.nvidia.com/fi-fi/")

Found 8 relevant links:


{'links': [{'type': 'about page',
   'url': 'https://www.nvidia.com/en-eu/about-nvidia/'},
  {'type': 'careers page',
   'url': 'https://www.nvidia.com/en-eu/about-nvidia/careers/'},
  {'type': 'partners',
   'url': 'https://www.nvidia.com/en-eu/about-nvidia/partners'},
  {'type': 'company policies',
   'url': 'https://www.nvidia.com/en-eu/about-nvidia/company-policies'},
  {'type': 'foundation', 'url': 'https://www.nvidia.com/en-us/foundation/'},
  {'type': 'research', 'url': 'https://www.nvidia.com/en-us/research/'},
  {'type': 'corporate social responsibility',
   'url': 'https://www.nvidia.com/en-us/csr/'},
  {'type': 'investor relations',
   'url': 'https://investor.nvidia.com/home/default.aspx'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [30]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [33]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

select_relevant_links for https://huggingface.co by calling the gpt-5-nano...
Found 10 relevant links:
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
about 7 hours ago
•
787k
•
3.62k
mistralai/Mistral-Medium-3.5-128B
Updated
2 days ago
•
16.6k
•
279
SulphurAI/Sulphur-2-base
Updated
about 12 hours ago
•
55.5k
•
270
openai/privacy-filter
Updated
14 days ago
•
155k
•
1.31k
SeeSee21/Z-Anime
Updated
9 days ago
•
3.82k
•
175
Browse 2M+ models
Spaces
Running
on
Zero
MCP
951
Wan2.2 14B Fast Preview
🐌
951
generate a video from an image with a text prompt
Running
on
Zero
MCP
2.54k
Wan2.2 14B Preview
🐌
2.54k
generate a video from an image with a tex

In [37]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [35]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [38]:
get_brochure_user_prompt("Notarec", "https://notarec.fi/")

select_relevant_links for https://notarec.fi/ by calling the gpt-5-nano...
Found 16 relevant links:


'\nYou are looking at a company called: Notarec\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nRekrytointi ja HR-Palvelut | Kasvuyritysten Kumppani | Notarec\n\nSkip to main content\nHinnasto\nReferenssit\nShow submenu for Ratkaisut\nRatkaisut\nRekrytointipalvelut\nHR-palvelut\nValmennuspalvelut\nUravalmennus\nShow submenu for Toimialat\nToimialat\nHuipputeknologia\nAsiantuntijayritykset\nEnergia-ala\nTeollisuus\nShow submenu for Sisältöhubi\nSisältöhubi\nTyökalupakki\nBlogi & Ajankohtaista\nTapahtumat & Webinaarit\nShow submenu for Meistä\nMeistä\nYhteys\nNotarecista\nOlemme osa HTGP Groupia\nMeille töihin\nOpen main navigation\nClose main navigation\nHinnasto\nReferenssit\nShow submenu for Ratkaisut\nRatkaisut\nRatkaisut\nRatkaisut\nRekrytointipalvelut\nHR-palvelut\nValmennuspalvelut\nUravalmennus\nShow submenu for Toimialat\nToimialat\nToimia

In [40]:



def create_brochure(company_name, url):
    responce = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = responce.choices[0].message.content
    display(Markdown(result))

In [18]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

openai = OpenAI()

MODEL = 'gpt-5-nano'

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


def create_brochure(company_name, url):
    responce = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = responce.choices[0].message.content
    display(Markdown(result))


def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.
Make additional markdown section where you are analysing good things about the website and its content, and what could be improved.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

def select_relevant_links(url):
    print(f"select_relevant_links for {url} by calling the {MODEL}...")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links:")
    return links

### Tuotos

create_brochure("Notarec", "https://notarec.fi")

select_relevant_links for https://notarec.fi by calling the gpt-5-nano...
Found 18 relevant links:


# Notarec – Kasvuyritysten kumppani rekrytoinnissa, people & culture -kehittämisessä ja valmennuksissa

Oikeat osaajat kasvusi tarpeisiin - nopeasti, läpinäkyvästi ja ilman turhaa säätöä.

Notarec on osa HTGP Groupia ja mamma-viivaansa pitävä yhteistyökumppani teknologia-, energia- sekä teollisuusyrityksille, asiantuntijayrityksille ja energiateollisuuden toimijoille. Meidän motto: Not another recruiter. Not another HR. Not another coach. Meillä löydät kokonaisvaltaisen – ja tuloshakuisen – kumppanin kasvun tueksi.

---

## Palvelumme

- Rekrytointi & Suorahaku
  - Haastavista yksittäisistä rekrytoinneista massiivisiin rekrytointiprojekteihin. Etsimme oikeat osaajat nopeasti, läpinäkyvästi ja ilman turhia säätöjä.

- HR-palvelut
  - Strateginen HR-osaaminen projekteihin ja väliaikaisiin tarpeisiin – kestävää kulttuuria ja toimivia prosesseja tukemaan kasvua.

- Valmennus & Kehittäminen
  - Täsmäkoulutukset, esihenkilövalmennukset, johtoryhmäcoaching sekä kulttuurimuutosmatkat.

- Uravalmennus
  - Tukea urakehitykseen ja ammatilliseen kasvuun.

- Maksuttomat mahdollisuudet
  - Tee maksuton HR-kuntokartoitus – hyvä alku jokaiselle kasvutarinalle.

---

## Toimialat ja asiakkaat

- Toimialat: huipputeknologia, teollisuus, energia-ala sekä asiantuntijayritykset.
- Asiakkaat: teknologiayritykset, teollisuusyritykset, asiantuntijayritykset ja energiateollisuuden toimijat.
- Tavoitteena: oikeat osaajat kasvunne tueksi – nopeasti ja ilman turhaa säätöä.

---

## Yrityskulttuuri ja arvot

- Kasvukeskeinen kumppani: meidän tehtävämme on tehostaa kasvuprosessejasi.
- Kestävää kulttuuria: HR-osaaminen ja prosessit, jotka tukevat pitkäjänteistä kehitystä.
- Proof of performance: yli 1 000 onnistunutta rekrytointia ja NPS-tavoitteet (92) 2025 näkyvissä.
- Tiimityö ja osaaminen: tiivis porukka, joka jakaa kokemuksiaan ja tavoitteitaan.
- Notarec vs. “normaali rekrytoija”: Not another recruiter – meillä on laajempi ote ihmisistä, kulttuurista ja kehittämisestä.

---

## Miksi valita Notarec?

- Kokonaisvaltaisuus: yhdellä kumppanilla kaikki rekrytointi-, HR- ja valmennustarpeet.
- Läpinäkyvyys: suorat ratkaisut ilman turhaa byrokratiaa.
- Kasvun tuki: sekä teknologia- että perinteisemmät toimialat korkealla vauhdilla.
- Kansainvälinen ja kotimainen konteksti: osa HTGP Groupia, kuitenkin asiakaslähtöinen lähestymistapa.
- Aktiiviset työkalut: sisällöstä ja koulutuksista käsin tukenasi – sisältöhubin, työkalupakin ja tapahtumien kautta.

---

## Careers & Meille Töihin

- Meille töihin -painotus: etsimme lahjakkuuksia, jotka haluavat vaikuttaa kasvavien yritysten menestykseen.
- Notariaden sloganit: Not another recruiter, Not another HR, Not another coach – viestivät laajan ja integroidun osaamisen tavoitteen.
- Mahdollisuudet: Rekrytointi, HR-palvelut, valmennus ja uravalmennus – osaamisrata, joka tukee sekä UCB:tä että kasvun kannattajia.
- Käytännön: voit varata maksuttoman HR-kuntokartoituksen ja saada hyvä aloitus Notarecin kanssa.

---

## Tiedonlähteet ja lisäresurssit

- Hinnasto, Referenssit, Ratkaisut (rekrytointi, HR, valmennus, uravalmennus)
- Toimialat: Huipputeknologia, Energia-ala, Teollisuus, Asiantuntijayritykset
- Sisältöhubi: Työkalupakki, Blogi & Ajankohtaista, Tapahtumat & Webinaarit
- Yhteys, Notarecista, Olemme osa HTGP Groupia, Meille töihin

- CTA: Tee maksuton HR-kuntokartoitus

---

## Analyysi: Hyvät puolet sivustosta ja parannusehdotukset

- Hyvät puolet
  - Selvä, ytimekäinen arvolupaus: “Oikeat osaajat kasvusi tarpeisiin – nopeasti, läpinäkyvästi ja ilman turhaa säätöä.”
  - Selkeä portfolio: rekrytointi, HR-palvelut, valmennus, uravalmennus – kaikki oleellinen löytyy yhdestä paikasta.
  - Toimialat ja asiakkaat ovat konkreettisia, mikä auttaa kohdentamista (teknologia, energia, teollisuus, asiantuntijayritykset).
  - Konkreettiset mittarit: NPS tavoite, 1,000+ onnistunutta rekrytointia, 10 kovaa ammattilaista. Luovat uskottavuutta.
  - Kulttuuriviesti: Not “normaali rekrytoija” – erottuva, moderni ja ihmiskeskeinen ote.
  - CTA-keskeisyys: maksuton HR-kuntokartoitus näkyvissä, helposti aloitetaan.

- Parannusehdotukset
  - Visuaalinen johdonmukaisuus: jos sivuston kieli on pääosin suomi, varmista samankaltainen termistö ja typografia koko sivustolla sekä navigaatiossa.
  - Esimerkit ja referenssit: lisää konkreettisia case-studies tai lyhyitä referenssejä (mitä ongelmaa ratkaistiin, tulokset, mitkä palvelut käytettiin).
  - Haku ja navigointi: paranna hakutoimintojen näkyvyyttä (esim. “Tee maksuton HR-kuntokartoitus” -kutsun sijoittelu ja näkyvyys useammassa kohtaa).
  - Asiakaskokemukset: lisää video- tai lyhyet tekstitodistukset asiakkaista, jotka kertovat lyhyesti kuinka Notarec auttoi.
  - Careers-sisällöt: avointen työpaikkojen osio sekä urapolut – voisivat olla paremmin esillä (esim. “Hae nyt” -nappulat joka sivulla).
  - Kansainvälinen vs. kotimainen: jos yritys palvelee ulkomailla tai kansainvälisiä asiakkaita, lisää selkeästi, miten toimitaan monikielisesti ja miten kansainväliset projektit hoidetaan.
  - Tiivis tarinankerronta: lisää Notarecin tarina – miksi perusti Notarecin, mitkä ovat sen kaupunkikuvat ja tiimin intohimot – tuo henkilöstö ihmisten tarinana esiin.

---

Jos haluat, voin tehdä tiiviin, visuaalisen version tästä brutschurista (brochure) valmiiksi printti-/pdf-formaattiin, tai räätälöidä sen erityisesti hakukoneoptimoiduksi verkkokäyttöön.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [16]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [17]:
stream_brochure("Notarec", "https://notarec.fi")

select_relevant_links for https://notarec.fi by calling the gpt-5-nano...
Found 13 relevant links:


# Welcome to Notarec: The HR Superheroes You Didn’t Know You Needed

---

## Who are we?  
Notarec isn’t your typical recruiter — nope, we’re *not another recruiter.* We’re your full-stack HR sidekick, your people & culture whisperers, and your coaching champions. Based in Finland, proudly part of the HTGP Group, we specialize in turbocharging growth companies by finding the **right talent**—quickly, transparently, and without the usual chaos.

---

## What do we do?  
We blend brains and heart in three powerful ways:

- **Rekrytointi & Suorahaku (Recruitment & Direct Search):** From snagging that elusive game-changer to orchestrating entire recruitment festivals, we get the right folks on your team.
  
- **People & Culture (HR Services):** Strategic HR wizardry for projects or interim gigs. We build sustainable cultures and ironclad processes that fuel your company’s growth engine.
  
- **Valmennus & Kehittäminen (Coaching & Development):** Tailored training, leadership coaching, and culture transformation journeys. We’re here to sharpen skills and elevate your teams.

---

## Who do we work with?  
If you’re in:

- High-tech innovation  
- Expert consultancy firms  
- Energy sector pioneers  
- Industrial masterminds  

...then congrats — we might just be your new best friend.

---

## Why choose Notarec?  
- We’ve rocked **1000+ challenging recruitments** (we hear that’s a bit of a flex).  
- Boasting a stellar **NPS of 92 for 2025**; our clients love us, and we love them back.  
- A lean, mean team of **10 absolute HR pros** who live and breathe people power.  

---

## Culture that clicks  
At Notarec, we don’t just do HR — we are HR. Imagine a place where your insights fuel growth, authenticity rules, and humor lightens the hardest of days (because let’s face it, hiring is serious business but who says we can’t have fun?).  

---

## Thinking career?  
If you're itching to join a crew where expertise meets empathy, where your skills don’t gather dust but set the training room on fire, check out our “Meille töihin” (Careers) page. We’re growing, and with us, “boring” HR jobs are a myth.

---

## Bonus! Free HR Check-up  
Not sure how your HR health looks? We offer a **free HR-kuntokartoitus**—a little wellness check for your workplace culture and HR processes. Because prevention is better than scrambling for a last-minute recruitment rush!

---

## TL;DR:  
**Notarec = The recruitment & HR Avengers (minus the capes).** We turn people into culture, culture into growth, and growth into success. Ready to stop wasting time and start building dream teams without the drama? Join the Notarec family or let us join *your* team-building adventures!

---

For a hassle-free chat and to see what makes us tick, hop on over to: [Notarec Homepage](https://www.notarec.fi)

---

*Notarec: Not just another recruiter. Your partner in people power.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

In [ ]:
## teee oma business use case tähän

